# Attention and Transformers Walkthrough

This notebook mirrors the Hugging Face fine-tuning cells from the attention and Transformers chapter. It is meant for interactive inspection: run one cell at a time, check token IDs, masks, label shapes, batch shapes, metrics, and a small training loop.

By default, the notebook uses synthetic toxic-comment-style rows and a tiny Hugging Face checkpoint so the pipeline can be checked quickly. For a real run, place the Kaggle `train.csv` file in `data/train.csv` and set `ATTN_CHECKPOINT=distilbert-base-uncased` before starting the notebook kernel.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Import Libraries and Set Run Constants

Keep the checkpoint, seed, maximum length, batch sizes, and package versions in the run record. The defaults are intentionally small; they are for learning the workflow, not for reporting final assignment numbers.

In [ ]:
import os
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, roc_auc_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

LABEL_COLUMNS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]
TEXT_COLUMN = "comment_text"

SEED = int(os.environ.get("ATTN_SEED", "1234"))
CHECKPOINT = os.environ.get(
    "ATTN_CHECKPOINT", "hf-internal-testing/tiny-random-distilbert"
)
MAX_LENGTH = int(os.environ.get("ATTN_MAX_LENGTH", "64"))
EPOCHS = float(os.environ.get("ATTN_EPOCHS", "1"))
BATCH_SIZE = int(os.environ.get("ATTN_BATCH_SIZE", "4"))
EVAL_BATCH_SIZE = int(os.environ.get("ATTN_EVAL_BATCH_SIZE", "8"))
LIMIT_TRAIN = int(os.environ.get("ATTN_LIMIT_TRAIN", "96"))
LIMIT_VAL = int(os.environ.get("ATTN_LIMIT_VAL", "32"))
LOCAL_FILES_ONLY = os.environ.get("ATTN_LOCAL_FILES_ONLY", "0") == "1"
USE_FP16 = torch.cuda.is_available() and os.environ.get("ATTN_FP16", "0") == "1"

set_seed(SEED)

print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("Checkpoint:", CHECKPOINT)
print("CUDA available:", torch.cuda.is_available())

### Reader Checkpoint

After the previous cell runs, confirm that the printed paths, shapes, commands, or tables match the section description before moving on. If this checkpoint fails in a public-repo environment, fix dependencies, data paths, or guarded flags before starting longer runs.

## 2. Load Kaggle Data or Synthetic Data

The Kaggle data is not bundled with the repository. If `data/train.csv` is missing, this cell creates a small synthetic multi-label dataset so the rest of the notebook can still run.

In [ ]:
def make_synthetic_frame(repeats=32):
    templates = [
        ("thanks for the careful explanation", [0, 0, 0, 0, 0, 0]),
        ("this is toxic and insulting", [1, 0, 0, 0, 1, 0]),
        ("obscene toxic insult in a long argument", [1, 0, 1, 0, 1, 0]),
        ("a direct threat appears in the message", [1, 0, 0, 1, 0, 0]),
        ("identity hate and obscene abuse", [1, 0, 1, 0, 0, 1]),
        ("severe toxic attack with threat and insult", [1, 1, 0, 1, 1, 0]),
        ("ordinary disagreement about the topic", [0, 0, 0, 0, 0, 0]),
        ("neutral reference to identity without abuse", [0, 0, 0, 0, 0, 0]),
    ]
    rows = []
    for repeat in range(repeats):
        for text, labels in templates:
            row = {TEXT_COLUMN: f"{text} example {repeat}"}
            row.update(dict(zip(LABEL_COLUMNS, labels)))
            rows.append(row)
    return pd.DataFrame(rows)

data_path = Path("data/train.csv")
force_synthetic = os.environ.get("ATTN_FORCE_SYNTHETIC", "0") == "1"

if data_path.exists() and not force_synthetic:
    frame = pd.read_csv(data_path)
    dataset_source = str(data_path)
else:
    frame = make_synthetic_frame()
    dataset_source = "synthetic"

required_columns = {TEXT_COLUMN, *LABEL_COLUMNS}
missing_columns = required_columns - set(frame.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

frame = frame[[TEXT_COLUMN, *LABEL_COLUMNS]].copy()
print("Dataset source:", dataset_source)
print("Rows:", len(frame))
print(frame[LABEL_COLUMNS].sum().sort_values(ascending=False))
frame.head()

## 3. Create a Reproducible Train-Validation Split

Model selection should use the validation split. Do not repeatedly use the official Kaggle test leaderboard as a validation set.

In [ ]:
rng = np.random.default_rng(SEED)
indices = np.arange(len(frame))
rng.shuffle(indices)

validation_fraction = 0.2
val_count = max(1, int(round(len(indices) * validation_fraction)))
val_indices = indices[:val_count]
train_indices = indices[val_count:]

train_frame = frame.iloc[train_indices].reset_index(drop=True).head(LIMIT_TRAIN)
val_frame = frame.iloc[val_indices].reset_index(drop=True).head(LIMIT_VAL)

print("Train rows:", len(train_frame))
print("Validation rows:", len(val_frame))
print("Train positives:")
print(train_frame[LABEL_COLUMNS].sum())
print("Validation positives:")
print(val_frame[LABEL_COLUMNS].sum())

## 4. Inspect Tokenization

Tokenization is preprocessing. Keep the checkpoint, tokenizer, truncation policy, padding policy, and maximum length consistent across splits.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT,
    local_files_only=LOCAL_FILES_ONLY,
)

example = "The movie was not good."
encoded = tokenizer(example, truncation=True, max_length=32)

print("Keys:", encoded.keys())
print("Input IDs:", encoded["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("Attention mask:", encoded["attention_mask"])

## 5. Convert Rows to PyTorch Datasets

Each row has one text string and six independent binary labels. The model should see a floating-point label vector, not one softmax class ID.

In [ ]:
class CommentDataset(torch.utils.data.Dataset):
    def __init__(self, data_frame, tokenizer, max_length):
        texts = data_frame[TEXT_COLUMN].astype(str).tolist()
        self.encodings = tokenizer(texts, truncation=True, max_length=max_length)
        self.labels = torch.tensor(
            data_frame[LABEL_COLUMNS].to_numpy(dtype=np.float32),
            dtype=torch.float32,
        )

    def __len__(self):
        return int(self.labels.shape[0])

    def __getitem__(self, index):
        item = {
            key: torch.tensor(value[index])
            for key, value in self.encodings.items()
        }
        item["labels"] = self.labels[index]
        return item


train_dataset = CommentDataset(train_frame, tokenizer, MAX_LENGTH)
val_dataset = CommentDataset(val_frame, tokenizer, MAX_LENGTH)

sample = train_dataset[0]
print("Sample keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels shape:", tuple(sample["labels"].shape))
print("labels:", sample["labels"].tolist())

## 6. Check Dynamic Padding and Batch Shapes

The data collator pads each batch to the longest example in that batch. This is usually faster than padding every example to the global maximum length.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
features = [train_dataset[i] for i in range(min(3, len(train_dataset)))]
batch = data_collator(features)

for key, value in batch.items():
    print(key, tuple(value.shape), value.dtype)

## 7. Build the Multi-Label Sequence Classifier

The classifier produces six logits. Binary cross-entropy with logits is the right loss because several labels can be true for the same comment.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(LABEL_COLUMNS),
    problem_type="multi_label_classification",
    local_files_only=LOCAL_FILES_ONLY,
    ignore_mismatched_sizes=True,
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Output labels:", model.config.num_labels)

## 8. Define Metrics

For multi-label classification, apply a sigmoid to each logit independently. Macro F1 exposes rare-label behavior; micro F1 pools all label decisions.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = sigmoid(logits)
    predictions = (probabilities >= 0.5).astype(int)

    metrics = {
        "macro_f1": f1_score(
            labels, predictions, average="macro", zero_division=0
        ),
        "micro_f1": f1_score(
            labels, predictions, average="micro", zero_division=0
        ),
    }
    try:
        metrics["macro_roc_auc"] = roc_auc_score(
            labels, probabilities, average="macro"
        )
    except ValueError:
        metrics["macro_roc_auc"] = float("nan")
    return {key: float(value) for key, value in metrics.items()}


zero_logits = np.zeros((len(val_dataset), len(LABEL_COLUMNS)))
validation_labels = val_frame[LABEL_COLUMNS].to_numpy(dtype=np.float32)
print(compute_metrics((zero_logits, validation_labels)))

## 9. Train One Small Run

This is a pipeline check. For assignment results, use a real checkpoint, the Kaggle CSV, and a controlled comparison through `toxic_comments_transformer.py` so the run writes artifacts.

In [ ]:
def make_training_arguments(output_dir):
    kwargs = {
        "output_dir": output_dir,
        "learning_rate": 2e-5,
        "per_device_train_batch_size": BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "num_train_epochs": EPOCHS,
        "weight_decay": 0.01,
        "save_strategy": "no",
        "logging_strategy": "epoch",
        "report_to": [],
        "seed": SEED,
        "fp16": USE_FP16,
    }
    try:
        return TrainingArguments(eval_strategy="epoch", **kwargs)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **kwargs)


training_args = make_training_arguments("runs/notebook-toxic-transformer")
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

try:
    trainer = Trainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = Trainer(tokenizer=tokenizer, **trainer_kwargs)

started_at = time.perf_counter()
train_output = trainer.train()
train_seconds = time.perf_counter() - started_at
validation_metrics = trainer.evaluate()

print("Training loss:", train_output.training_loss)
print("Training seconds:", round(train_seconds, 2))
print(validation_metrics)

## 10. Inspect Probabilities and Error Counts

Use probabilities for threshold analysis and ROC-AUC. When writing an error analysis, avoid reproducing raw offensive text; paraphrase or anonymize examples when needed.

In [ ]:
pred_output = trainer.predict(val_dataset)
probabilities = sigmoid(pred_output.predictions)
predictions = (probabilities >= 0.5).astype(int)
labels = val_frame[LABEL_COLUMNS].to_numpy(dtype=np.int32)

for index, label_name in enumerate(LABEL_COLUMNS):
    false_positives = int(((predictions[:, index] == 1) & (labels[:, index] == 0)).sum())
    false_negatives = int(((predictions[:, index] == 0) & (labels[:, index] == 1)).sum())
    print(f"{label_name:14s} false positives={false_positives:2d} false negatives={false_negatives:2d}")

probability_frame = pd.DataFrame(
    probabilities,
    columns=[f"p_{label}" for label in LABEL_COLUMNS],
)
pd.concat(
    [val_frame[[TEXT_COLUMN, *LABEL_COLUMNS]].reset_index(drop=True), probability_frame],
    axis=1,
).head()

## 11. Move From Notebook Check to Reproducible Scripts

Use the scripts for reportable runs because they save the command, versions, metrics, runtime, and CSV/JSON artifacts.

```sh
poetry run python toxic_comments_transformer.py --quick --save-artifacts
poetry run python toxic_comments_tfidf_baseline.py --data-dir data --save-artifacts
poetry run python toxic_comments_data_audit.py --data-dir data --max-lengths 128 256 512 --save-artifacts
poetry run python attention_scaling_benchmark.py --lengths 64 128 256 512 --save-artifacts
```

For the homework, keep the same split for the baseline, Transformer run, and controlled comparison unless the comparison is explicitly about data splitting.

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.